

### 4 End Marks ###


In [25]:
from pathlib import Path
import os
import pandas as pd

# Colors are defined only here as (R, G, B) tuples
ColorPrimary = (220,120,0)
ColorSecondary = (0,180,220)

# Hex versions for matplotlib/seaborn, derived from the tuples above
color_primary = '#%02X%02X%02X' % ColorPrimary
color_secondary = '#%02X%02X%02X' % ColorSecondary

# output_notebook(resources=INLINE) # --- BOKEH JUPYTERLAB SETUP ---

# Dataframes for calculations
# allDataDF
# filteredDataDF
# firstMarksDF

#### 4-1 Filter double marks. ####

Make slider to interactively adjust the threshold for the filter.
Attention: the current filter method takes away all marks set in the beginning. timecode 0 + timewindow is erased. Supposedly not a problem (or even favorable), but the fact needs to be kept in mind.

The first step is to eliminate subsequent marks of single individuals, if they were made within a certain time frame (time window, e.g. 5000ms).
This needs to be done carefully, as we don't want to eliminate marks that would already indicate a moment where the participant actually would have wanted to indicate their next guess of an ending.


In [26]:
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from ipywidgets import interact, IntSlider

# Load the pandas dataframe containing the marks data.
# It was created in the previous notebook "01-Participants-Background".
allDataDF = pd.read_parquet("../data/notebookData/allDataDF.parquet")

# Create an output area to hold the chart
plot_output = widgets.Output()

# Keep a reference to the current figure so the save button can export it
current_fig = None
current_window = None

def filter_marks(allData, time_diff):
    filtered_data_list = []
    filtered = allData.groupby('userID')
    for userID, user_data in filtered:
        last_mark_TC = 0
        filtered_marks = []
        for _, row in user_data.iterrows():
            current_time = row["timeStamp"]
            if current_time - last_mark_TC >= time_diff:
                filtered_marks.append(row)
                last_mark_TC = current_time
        if filtered_marks:
            filtered_data_list.append(pd.DataFrame(filtered_marks))

    if filtered_data_list:
        return pd.concat(filtered_data_list, ignore_index=True)
    return pd.DataFrame(columns=allData.columns)

# Pre-calculate the curve data once to keep the slider responsive
curve_data = []
for i in range(0, 70001, 1000):
    m = filter_marks(allDataDF, i)
    curve_data.append({'windowSize': i, 'marks': len(m)})
df_curve = pd.DataFrame(curve_data)

def update_chart(f_value):
    global current_fig, current_window
    with plot_output:
        global filteredDataDF
        filteredDataDF = filter_marks(allDataDF, f_value)

        plot_output.clear_output(wait=True)

        # Calculate current point
        current_marks = len(filteredDataDF)

        # Create the figure
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.set_theme(style="whitegrid") # Seaborn styling
        #sns.set_theme(style="ticks", font_scale=0.75)

         # Plot the curve using Seaborn
        sns.lineplot(data=df_curve, x='windowSize', y='marks', linewidth=0.6, color=color_secondary, ax=ax, label='Mark Trend')
        sns.scatterplot(data=df_curve, x='windowSize', y='marks', marker="o", color="black", s=14, ax=ax)

        # Single-point dataframe for the highlight to stay consistent with Seaborn patterns
        highlight_df = pd.DataFrame({'windowSize': [f_value], 'marks': [current_marks]})
        sns.scatterplot(data=highlight_df, x='windowSize', y='marks', color=color_primary, s=200, ax=ax, zorder=5, label=f'Current: {current_marks}')

        # Formatting
        ax.set_title('Window Size vs Number of Marks', fontsize=12, fontweight='normal')
        ax.set_xlabel('Window Size (ms)', fontsize=10)
        ax.set_ylabel('Number of Marks', fontsize=10)
        ax.set_xlim(0, 70000)
        ax.set_ylim(0, df_curve['marks'].max() + 20)
        ax.grid(True, alpha=0.3)
        ax.legend()

        current_fig = fig
        current_window = f_value
        save_status.value = ''

        plt.show()

### Export the figure (only when the button is pressed)
image_output_path = Path("../image-output") # Directory for exported images

def save_svg(_):
    if current_fig is None:
        save_status.value = 'No plot to save yet.'
        return
    try:
        image_output_path.mkdir(exist_ok=True)
        svg_filename = image_output_path / f"windowSizeVsNumberOfMarks_{current_window}ms.svg"
        current_fig.savefig(svg_filename, format="svg", bbox_inches="tight")
        save_status.value = f"Saved SVG: {svg_filename}"
    except Exception as e:
        save_status.value = f"SVG export failed: {e}"

save_button = widgets.Button(description='Save SVG', icon='download')
save_status = widgets.Label()
save_button.on_click(save_svg)

### Save filteredDataDF for use in other notebooks (only when the button is pressed)
notebook_data_path = Path("../data/notebookData")

def save_parquet(_):
    try:
        notebook_data_path.mkdir(parents=True, exist_ok=True)
        parquet_filename = notebook_data_path / "filteredDataDF.parquet"
        filteredDataDF.to_parquet(parquet_filename)
        save_status.value = f"Saved parquet ({len(filteredDataDF)} marks, {current_window}ms window): {parquet_filename}"
    except Exception as e:
        save_status.value = f"Parquet export failed: {e}"

save_parquet_button = widgets.Button(description='Save Data', icon='database')
save_parquet_button.on_click(save_parquet)

# Setup Slider
filter_slider = IntSlider(
    value=10000,
    min=1000,
    max=60000,
    step=1000,
    description='Time window:',
    continuous_update=False
)

# Link the slider to the update function
widgets.interactive(update_chart, f_value=filter_slider)

# Display the UI
display(widgets.HBox([filter_slider, save_button, save_parquet_button, save_status]))
display(plot_output)

# Trigger the initial draw
update_chart(filter_slider.value)

Output()

#### 4-2. Display Marks in a Timeline ####

In [27]:
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML

# Set the renderer for PyCharm
pio.renderers.default = "notebook_connected"

# Export button for a Plotly figure (saves only when pressed)
def make_svg_button(fig, filename):
    button = widgets.Button(description='Save SVG', icon='download')
    status = widgets.Label()

    def save(_):
        try:
            image_output_path.mkdir(exist_ok=True)
            svg_filename = image_output_path / filename
            fig.write_image(svg_filename, format="svg", width=1200, height=800, scale=1.5)
            status.value = f"Saved SVG: {svg_filename}"
        except Exception as e:
            status.value = f"SVG export failed: {e}"

    button.on_click(save)
    return widgets.HBox([button, status])

# 1. Chart displaying all marks in one timeline
# ------------------------------------------------------

if not filteredDataDF.empty:
    # We add a constant column for the y-axis to align all points
    plot_df = filteredDataDF.copy()
    plot_df['y_pos'] = 1

    fig1 = px.scatter(
        plot_df,
        x='timeStamp',
        y='y_pos',
        color_discrete_sequence=[f'rgb{ColorPrimary}'],
        title='User Timeline',
        labels={'timeStamp': 'Time', 'y_pos': ''},
        opacity=0.3
    )

    # Customize rollover (hover) and markers
    fig1.update_traces(
        marker=dict(size=12),
        hovertemplate="<b>Participant ID</b>: %{customdata[0]}<br><b>Timestamp</b>: %{x}<extra></extra>",
        customdata=plot_df[['userID']]
    )

    # Clean up Y-axis as it's just a single line
    fig1.update_yaxes(showticklabels=False, showgrid=False, range=[0.5, 1.5])
    fig1.update_layout(width=1200, height=400, template='plotly_white', xaxis_range=[0, None])

    ### Display figure 1 with its export button
    fig1.show()
    display(make_svg_button(fig1, "allMarksTimeline.svg"))

# 2. Chart displaying marks by participants separate
# ------------------------------------------------------

if not filteredDataDF.empty:
    unique_user_ids = sorted(filteredDataDF['userID'].unique())
    user_positions = {user_id: i for i, user_id in enumerate(unique_user_ids)}

    plot_df_sep = filteredDataDF.copy()
    plot_df_sep['y_pos'] = plot_df_sep['userID'].map(user_positions)

    fig2 = px.scatter(
        plot_df_sep,
        x='timeStamp',
        y='y_pos',
        title="Timeline of Marks by Participants",
        labels={'timeStamp': 'Time (ms)', 'y_pos': 'Participants'},
        color_discrete_sequence=[f'rgb{ColorPrimary}'],
        opacity=0.6
    )

    # Customize appearance and hover
    fig2.update_traces(
        marker=dict(size=10, line=dict(width=1, color='white')),
        hovertemplate="<b>Participant ID</b>: %{customdata[0]}<br><b>Timestamp</b>: %{x}<extra></extra>",
        customdata=plot_df_sep[['userID']]
    )

    # Map the Y-axis ticks to show the actual User IDs instead of indices
    fig2.update_layout(
        width=1200,
        height=800,
        template='plotly_white',
        xaxis_range=[0, None],
        yaxis=dict(
            tickmode='array',
            tickvals=list(range(len(unique_user_ids))),
            ticktext=[str(int(uid)) for uid in unique_user_ids],
            range=[-1, len(unique_user_ids)]
        )
    )

    ### Display figure 2 with its export button
    fig2.show()
    display(make_svg_button(fig2, "allIndividualMarksTimeline.svg"))


    # Calculate number of marks per participant
    marks_per_participant = filteredDataDF.groupby('userID').size()

    # Count participants by mark frequency
    participants_with_1_mark = (marks_per_participant == 1).sum()
    participants_with_2_or_more_marks = (marks_per_participant >= 2).sum()

    # Calculate percentages
    total_participants = len(marks_per_participant)

    percentage_1_mark = participants_with_1_mark / total_participants * 100
    percentage_2_or_more_marks = participants_with_2_or_more_marks / total_participants * 100

    frequencyDF = pd.DataFrame(
        {'N': [participants_with_1_mark, participants_with_2_or_more_marks, total_participants],
         '%': [percentage_1_mark, percentage_2_or_more_marks, 100.0]},
        index=pd.Index(['1 mark', '2 or more marks', 'Total'], name='Participants with')
    )

    # Calculate percentage of all remaining marks before 611.5 seconds
    timecode_limit = 611_500  # 611.5 seconds, in milliseconds

    total_remaining_marks = len(filteredDataDF)
    remaining_marks_before_limit = (filteredDataDF['timeStamp'] < timecode_limit).sum()
    percentage_remaining_marks_before_limit = (
        remaining_marks_before_limit / total_remaining_marks * 100
    )

    beforeLimitDF = pd.DataFrame(
        {'N': [remaining_marks_before_limit, total_remaining_marks],
         '%': [percentage_remaining_marks_before_limit, 100.0]},
        index=pd.Index(['Before 611.5 s', 'Total'], name='Remaining marks')
    )

    # Show both tables side by side underneath the charts
    tables = [
        frequencyDF.style.format({'%': '{:.1f}'}).set_caption('Participant mark frequency'),
        beforeLimitDF.style.format({'%': '{:.1f}'}).set_caption('Remaining marks before 611.5 s'),
    ]
    display(HTML(
        '<div style="display: flex; gap: 2em; align-items: flex-start;">'
        + ''.join(f'<div>{t.to_html()}</div>' for t in tables)
        + '</div>'
    ))


,N,%
Participants with,,
1 mark,20,40.8
2 or more marks,29,59.2
Total,49,100.0
,N,%
Remaining marks,,
Before 611.5 s,47,48.0
Total,98,100.0


#### 4.3 Use only the first mark of each participant. ####

In [28]:
import plotly.express as px

# Set a threshold for the first mark; filters out "accidental" clicks at the start of the session.
threshold = 10000

# Keep only the first mark for each user that is above the threshold
# Creates a clean list of the "initial guesses" for every participant,
# ensuring we only have one data point per person.

#  "sort" puts all remaining marks in chronological order.
#  "groupby('userID')" groups the data by each individual user and picks only
#  the very first mark they made after the threshold.
#  .reset_index(): This turns the data back into a standard table format (DataFrame)
#  so we can easily use it for plotting or further analysis.
firstMarksDF = allDataDF[allDataDF['timeStamp'] >= threshold].sort_values('timeStamp').groupby('userID').first().reset_index()

# Prepare the data for Bokeh
# Extracts every unique userID from your filtered "first marks" dataset.
# Creates a dictionary (a lookup table) that maps each User ID to a simple number (0, 1, 2, 3...).
# In plotting libraries like Bokeh, the Y-axis usually needs numerical coordinates. By doing this, you're saying: "User #110 should be drawn on the 1st row, User #106 on the 2nd row," and so on.
unique_user_ids_first = firstMarksDF['userID'].unique()
user_positions_first = {user_id: i for i, user_id in enumerate(unique_user_ids_first)}

# Create the list of colors in Hex/RGB for Plotly
# Assuming ColorPrimary/Secondary are (R, G, B) tuples
def to_rgb_str(tpl): return f'rgb{tpl}'

firstMarksDF['color'] = [
    to_rgb_str(ColorSecondary) if 100 <= uid <= 113 else to_rgb_str(ColorPrimary)
    for uid in firstMarksDF['userID']
]
firstMarksDF['y_pos'] = firstMarksDF['userID'].map(user_positions_first)

# --- Plotting ---
fig_first = px.scatter(
    firstMarksDF,
    x='timeStamp',
    y='y_pos',
    color='color',
    color_discrete_map="identity", # Use the strings in the 'color' column directly
    hover_name="userID",
    # custom_data lets Plotly split the IDs per color trace, so each point keeps its own ID
    custom_data=['userID'],
    #title="Timeline of First Marks by Participants",
    #labels={'timeStamp': 'Time (ms)', 'y_pos': 'Users'},
    opacity=0.6
)

# Customize Hover and Markers
fig_first.update_traces(
    marker=dict(size=10, line=dict(width=1, color='white')),
    hovertemplate="<b>User ID</b>: %{customdata[0]}<br><b>Timestamp</b>: %{x}<extra></extra>"
)

# Customize Axes
fig_first.update_layout(
    width=1200,
    height=800,
    template='plotly_white',
    xaxis_range=[0, None],
    showlegend=False,
    yaxis=dict(
        tickmode='array',
        tickvals=list(range(len(unique_user_ids_first))),
        ticktext=[str(int(uid)) for uid in unique_user_ids_first],
        range=[-1, len(unique_user_ids_first)]
    )
)

### Display the figure with its export button (make_svg_button is defined in 4-2)
fig_first.show()
display(make_svg_button(fig_first, "firstMarksTimeline.svg"))

# Calculate percentage of first marks before 611.5 seconds
timecode_limit = 611_500  # 611.5 seconds, in milliseconds

total_first_marks = len(firstMarksDF)
first_marks_before_limit = (firstMarksDF['timeStamp'] < timecode_limit).sum()
percentage_before_limit = first_marks_before_limit / total_first_marks * 100

firstBeforeLimitDF = pd.DataFrame(
    {'N': [first_marks_before_limit, total_first_marks],
     '%': [percentage_before_limit, 100.0]},
    index=pd.Index(['Before 611.5 s', 'Total'], name='First marks')
)
display(firstBeforeLimitDF.style.format({'%': '{:.1f}'}).set_caption('First marks before 611.5 s'))

,N,%
First marks,,
Before 611.5 s,25,51.0
Total,49,100.0


#### 4.4 Kernel density estimation (KDE) for the marks distribution ####

Cluster algorithms are not the best pick for 1-D data!
"The kernel density estimator is a non-parametric way to estimate the probability density function of a random variable." The random variable is the timecode of the marks provided by the participants. The variable is distributed over the time axis starting at 0 and ending at the end of the performance.

Good website on the topic: https://towardsdatascience.com/the-math-behind-kernel-density-estimation-5deca75cba38/

Documentation Python implementation: https://kdepy.readthedocs.io/en/latest/index.html




In [29]:
from KDEpy import *
from KDEpy import FFTKDE
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown
import plotly.graph_objects as go
import plotly.io as pio
from scipy.signal import find_peaks
from scipy.interpolate import interp1d

# Create an explicit output area for the Plotly chart
plot_output_kde = widgets.Output()

# Set the renderer for PyCharm
pio.renderers.default = "notebook_connected"

# Keep a reference to the current figure so the save button can export it
current_kde_fig = None
current_bw = None
current_dataset = None

# Datasets available for the KDE, looked up by name at plot time
# (so filteredDataDF always reflects the current time window slider in 4-1)
kde_dataset_names = ['firstMarksDF', 'allDataDF', 'filteredDataDF']

def update_plot(bw_value, dataset_name):
    global KernelWindowSize, current_kde_fig, current_bw, current_dataset, data
    KernelWindowSize = bw_value

    # Extract the timeStamp column of the selected dataset
    data = globals()[dataset_name]['timeStamp'].values

    with plot_output_kde:
        # Recalculate KDE
        try:
            # Recalculate KDE
            # The grid must extend beyond every data point (allDataDF contains marks at 0 ms)
            x = np.linspace(
                min(0, data.min() - KernelWindowSize),
                data.max() + KernelWindowSize,
                4096
            )

            y = FFTKDE(
                kernel='epa',
                bw=KernelWindowSize
            ).fit(data).evaluate(x)

            print(f"KDE calculated: x range {x.min():.1f}–{x.max():.1f} ms, y max {y.max():.8f}")

        except Exception as e:
            plot_output_kde.clear_output(wait=True)
            print("KDE calculation failed:")
            print(e)
            return


        # 2. Calculate Slope (Numerical Gradient)
        dy_dx = np.gradient(y, x)

        # Calculate peaks (local maxima)
        peaks, _ = find_peaks(y)
        peak_x = x[peaks]
        peak_y = y[peaks]

        # Calculate peaks of the slope / first derivative
        slope_peak_min_height = dy_dx.max() * 0.2
        slope_peak_min_prominence = dy_dx.max() * 0.1

        # Find the 3 highest density peaks for labels/markers
        top_peak_count = 3
        top_peak_indices = np.argsort(peak_y)[-top_peak_count:]
        top_peak_x = peak_x[top_peak_indices]
        top_peak_y = peak_y[top_peak_indices]

        # Sort the top 3 peaks chronologically for cleaner display
        top_peak_order = np.argsort(top_peak_x)
        top_peak_x = top_peak_x[top_peak_order]
        top_peak_y = top_peak_y[top_peak_order]

        slope_peaks, _ = find_peaks(
            dy_dx,
            height=slope_peak_min_height,
            prominence=slope_peak_min_prominence,
            distance=20
        )
        slope_peak_x = x[slope_peaks]
        slope_peak_y = dy_dx[slope_peaks]

        # Convert x-axis values from milliseconds to seconds for plotting
        x_seconds = x / 1000
        peak_x_seconds = peak_x / 1000
        top_peak_x_seconds = top_peak_x / 1000
        slope_peak_x_seconds = slope_peak_x / 1000
        data_seconds = data / 1000

        # Keep only positive slope values for area fill
        dy_dx_positive = np.where(dy_dx > 0, dy_dx, 0)

        # Clear previous output
        plot_output_kde.clear_output(wait=True)

        # Create figure with secondary y-axis
        from plotly.subplots import make_subplots
        fig = make_subplots(specs=[[{"secondary_y": True}]])
        #fig = go.Figure()

        # 1. Add KDE Curve (Primary Y-Axis)
        fig.add_trace(go.Scatter(
            # x=x,
            x=x_seconds,
            y=y,
            mode='lines',
            name=f'KDE Density (bw={bw_value})',
            line=dict(color='#555555', width=2)
        ), secondary_y=False)

        # Add Slope Curve (Secondary Y-Axis)
        fig.add_trace(go.Scatter(
            # x=x,
            x=x_seconds,
            y=dy_dx,
            mode='lines',
            name='Slope (1st Deriv)',
            line=dict(color='black', width=1)
        ), secondary_y=True)

        # Add positive slope area fill
        fig.add_trace(go.Scatter(
            x=x_seconds,
            y=dy_dx_positive,
            mode='lines',
            name='Positive Slope Area',
            line=dict(width=0),
            fill='tozeroy',
            fillcolor=f'rgba{ColorSecondary + (0.5,)}',
            showlegend=False
        ), secondary_y=True)

        # Add Slope Peak Markers
        fig.add_trace(go.Scatter(
            x=slope_peak_x_seconds,
            y=slope_peak_y,
            mode='markers+text',
            name='Slope Peaks',
            text=[f"{val:.1f}s" for val in slope_peak_x_seconds],
            textposition="top center",
            marker=dict(
                color='white',
                symbol='circle',
                size=10,
                line=dict(color='black', width=2)
            )
        ), secondary_y=True)

        # Add zero-line for the slope to easily see zero-crossings
        # fig.add_shape(type="line", x0=min(x_seconds), x1=max(x_seconds), y0=0, y1=0,
        #              line=dict(color="blue", width=1, dash="dash"),
        #              secondary_y=True)

        # Add Peak Markers
        fig.add_trace(go.Scatter(
            x=top_peak_x_seconds,
            y=top_peak_y,
            mode='markers+text',
            name='Top Density Peaks',
            text=[f"{val:.1f}s" for val in top_peak_x_seconds],
            textposition="top center",
            marker=dict(color='white', symbol='circle', size=10, line=dict(color='black', width=2))
        ), secondary_y=False)

        # 2. Add Rug plot (Individual Marks)
        fig.add_trace(go.Scatter(
            x=data_seconds,
            y=np.zeros_like(data_seconds),
            mode='markers',
            name='Marks by Participants',
            marker=dict(symbol='circle', color='black', size=10, opacity=0.4)
        ), secondary_y=False)

        # Update layout and axes
        fig.update_layout(
            title=f"KDE Density and Slope. Dataset: {dataset_name} ({len(data)} marks). Bandwidth: {bw_value}ms",
            width=1200, height=800,
            xaxis_title="Time (seconds)",
            # xaxis_range=[0, max(x_seconds)], # make sure x-axis starts at 0
            template='plotly_white',
            legend=dict(orientation="h", yanchor="bottom", y=-0.3, xanchor="center", x=0.5)
        )

        fig.update_xaxes(
            showgrid=True,
            zeroline=True
        )

        fig.update_yaxes(
            title_text="",
            showticklabels=True,
            tickmode="array",
            tickvals=[0],
            ticktext=["0"],
            ticks="",
            range=[-y.max() * 0.08, y.max() * 2.2],
            zeroline=True,
            showgrid=True,
            secondary_y=False
        )
        slope_abs_max = max(abs(dy_dx.min()), abs(dy_dx.max()))
        fig.update_yaxes(
            title_text="",
            showticklabels=True,
            tickmode="array",
            tickvals=[0],
            ticktext=["0"],
            ticks="",
            range=[-slope_abs_max * 2.2, slope_abs_max * 1.2],
            showgrid=False,
            zeroline=True,
            secondary_y=True
        )

        fig.add_annotation(
            text="Density (Anticipated Likelihood of Ending)",
            xref="paper",
            yref="paper",
            x=0.03,
            y=0.2,
            showarrow=False,
            textangle=-0
        )
        fig.add_annotation(
            text="1st Derivative of the Density Function ()",
            xref="paper",
            yref="paper",
            x=0.03,
            y=0.9,
            showarrow=False,
            textangle=-0
        )

        current_kde_fig = fig
        current_bw = bw_value
        current_dataset = dataset_name
        kde_save_status.value = ''

        ### Display the figure
        # plotly.js is loaded only once per kernel (in the first cell that shows a Plotly figure).
        # The widget Output can't use that copy, so load plotly.js into it again before each redraw.
        pio.renderers['notebook_connected'].activate()
        fig.show()

### Export the figure (only when the button is pressed)
def save_kde_svg(_):
    if current_kde_fig is None:
        kde_save_status.value = 'No plot to save yet.'
        return
    try:
        image_output_path.mkdir(exist_ok=True)
        svg_filename = image_output_path / f"kde_density_{current_dataset}_bw_{current_bw}.svg"
        current_kde_fig.write_image(svg_filename, format="svg", width=1200, height=800, scale=1.5)
        kde_save_status.value = f"Saved SVG: {svg_filename}"
    except Exception as e:
        kde_save_status.value = f"SVG export failed: {e}"

kde_save_button = widgets.Button(description='Save SVG', icon='download')
kde_save_status = widgets.Label()
kde_save_button.on_click(save_kde_svg)

# Make a selector to choose the dataset for the KDE
dataset_dropdown = Dropdown(
    options=kde_dataset_names,
    value='filteredDataDF',
    description='Dataset:'
)

# Make a slider to adjust the bandwidth of the KDE
bw_slider = IntSlider(
    value=10000,
    min=500,
    max=30000,
    step=500,
    description='Bandwidth:',
    continuous_update=False  # This is the key "event listener" for mouse release
)

#interact(update_plot, bw_value=bw_slider);

# Display everything together
display(widgets.HBox([dataset_dropdown, bw_slider, kde_save_button, kde_save_status]))
display(plot_output_kde)

# Link the dropdown and slider to the update function
widgets.interactive_output(update_plot, {'bw_value': bw_slider, 'dataset_name': dataset_dropdown})

# Trigger initial plot
update_plot(bw_slider.value, dataset_dropdown.value)

Output()

#### Bandwidth selection for KDE ####

A careful selection of the bandwidth parameter is a crucial step for kernel density estimation. Because our events on the timeline are not fully random but occur in relation to a specific music performance, we can derive guidelines and assumptions about what a suitable bandwidth is.

As the density of musical events, or the tempo of the musical performance, is rather slowly paced, rather long bandwidths are appropriate for the KDE algorithm. A larger bandwidth results in a smoother density estimate, which is appropriate for slow-paced musical performances where the events are spread out over a longer period of time.
In a music performance of fast tempo, a shorter bandwidth would be necessary, as even small gaps between marks could be related to different events or situations. (e.g. in tonal classical music, possible endings would typically be associated with tonic (I) chords which could occur several times within a few seconds (think for instance of Mozart's EXAMPLE).

BW = 10000ms: A bandwidth of 10 seconds shows two main peaks, or temporal zones, where participants reported expected endings in a particularly high density.

BW = 5000ms: When a bandwidth of only 5 seconds is applied, the kernel density estimation shows a more detailed representation, with smaller peaks and valleys. The first peak between 330 seconds and 400 seconds that was observed with a bandwidth of 10 seconds is differentiated into two smaller peaks, with a new main peak at around 380 seconds and a secondary peak at around 350 seconds. A new peak becomes visible at around 538 seconds.
The main peak at the end of the performance at ~625 becomes more pointed.
